# Module 5 — Hyperparameter Tuning

**By the end of this notebook, you will be able to:**
- Use `GridSearchCV` together with `GroupKFold` — the ready-made scikit-learn version of the cross-validation you built by hand in Module 3
- Explain why an untuned, more complex model can be *worse* than a simple one
- Read a hyperparameter search's results, not just its single "best" answer

**Context:** Module 4 left you with 8 RFE-selected columns and a `LinearRegression` baseline around RMSE 27.6. This module asks whether a more flexible model — `XGBRegressor` — can do better, and what happens if you use it without tuning it at all. `uv sync --group ml` is required for this module (`xgboost`).

## Rebuild the Module 4 pipeline

**Exercise:** Same sequence as Module 4 (`restrict_to_scope` → `columns_above_missing_threshold` → `drop_columns` → `fill_missing_by_city` → `add_temporal_features`), then reuse `selection.select_features_rfe` to get back the same 8 columns.

In [1]:
# TODO: rebuild enriched and the RFE-selected 8 columns, reusing what
# you already completed in Modules 2 and 4

from air_quality import data
from air_quality import features
from air_quality import evaluation
from air_quality import selection
from air_quality import tuning

from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
import xgboost as xgb


ALL_CITIES = ["Kampala", "Nairobi", "Lagos", "Bujumbura"]


train_df, _ = data.load_datasets()
df= data.restrict_to_scope(train_df,ALL_CITIES, train_df.columns)
fillable_columns = [c for c in df.columns if c not in ("city", "date")]


shitty_columns= data.columns_above_missing_threshold(df,0.7)
df_clean= data.drop_columns(df,shitty_columns)
df_clean= data.fill_missing_by_city(df_clean,fillable_columns,'city','date')
enriched = features.add_temporal_features(df_clean)
feature_cols = features.feature_columns(enriched)    

rfe_selected = selection.select_features_rfe(LinearRegression(),enriched,feature_cols,"pm2_5",8)

## What if you don't tune at all?

Try `XGBRegressor` with its defaults — no hyperparameters chosen deliberately — and evaluate it exactly like every other feature set this session, with `evaluate_group_cv`.

In [2]:
# Given: an untuned model, evaluated the same way as everything else this session
result_untuned = evaluation.evaluate_group_cv(xgb.XGBRegressor(random_state=42), enriched, rfe_selected, groups_col="city")
{k: v for k, v in result_untuned.items() if k != "folds"}

{'r2_mean': -0.7731593197523958,
 'rmse_mean': 31.880109856504554,
 'mae_mean': 16.70905422442226,
 'mae_std': 7.071758430025428,
 'rmse_std': 14.080315811467555,
 'r2_std': 1.648425834502944}

**Worse than Module 4's plain `LinearRegression` (RMSE ≈ 27.6) on the exact same 8 columns.** A more flexible model is not automatically better — without tuning, `XGBRegressor` has no reason to fit this particular, fairly small, fairly linear dataset well. Flexibility has to be pointed in the right direction by its hyperparameters, or it can hurt more than it helps.

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) does that search for you — and, combined with [`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) as its `cv`, it is the exact guarantee you built by hand in Module 3 (a group never appears in both train and validation for a fold), just applied automatically across every hyperparameter combination instead of manually across every city.

## Tune it properly

Open `src/air_quality/tuning.py` (new file, already scaffolded) and complete `tune_xgboost`. Its docstring and `tests/test_tuning.py` specify exactly what it should do. Run `uv run pytest tests/test_tuning.py -v` until it passes, then apply it here with a small grid — `{"max_depth": [3, 5], "learning_rate": [0.05, 0.2]}`, 4 combinations, nothing exhaustive.

In [3]:
# TODO: apply tune_xgboost once you have implemented it
param_grid= {"max_depth": [3, 5], "learning_rate": [0.05, 0.2]}
tuning_result = tuning.tune_xgboost(enriched,feature_cols=feature_cols,param_grid=param_grid, target_col='pm2_5', groups_col="city" )


In [4]:
# Given: every combination tried, not just the best one
import pandas as pd

pd.DataFrame(tuning_result["cv_results"]).sort_values("rmse")

,params,rmse,mae,r2
0,"{'learning_rate': 0.05, 'max_depth': 3}",27.442610,14.622092,0.070948
1,"{'learning_rate': 0.05, 'max_depth': 5}",29.881296,16.653535,-0.169970
3,"{'learning_rate': 0.2, 'max_depth': 5}",33.661017,20.602432,-0.766559
2,"{'learning_rate': 0.2, 'max_depth': 3}",38.822583,22.810837,-1.284965


Notice the table has `rmse`, `mae` and `r2` for every combination — the same three metrics `regression_metrics()` has always reported together. `GridSearchCV` still ranks combinations by `rmse` alone (`refit="rmse"` in `tune_xgboost`) since a search needs one number to compare against, but nothing stops you from reading the other two for the combination it picks.

**Question:** Compare four numbers: the untuned `XGBRegressor` above, the worst combination in this grid, the best combination, and Module 4's `LinearRegression` on the same 8 columns. Was it worth introducing a more complex model here? What does the gap between the best and worst combination in this small grid tell you about *why* tuning matters, independently of which model you picked?

## From notebook to pipeline

Update `run_advanced` in `src/air_quality/workflows.py` once more: after Module 4's feature selection, replace the model with `XGBRegressor` configured with the best parameters `tune_xgboost` found, and keep evaluating with `evaluate_group_cv`. As with every module since Module 2, there is no test for this — verify it by calling it below.

In [5]:
from air_quality.workflows import run_advanced

run_advanced()

{'best_params': {'learning_rate': 0.05, 'max_depth': 3},
 'best_rmse': np.float64(27.56736210003571),
 'cv_results': [{'params': {'learning_rate': 0.05, 'max_depth': 3},
   'rmse': np.float64(27.56736210003571),
   'mae': np.float64(14.244063456067169),
   'r2': np.float64(0.08450761609216201)},
  {'params': {'learning_rate': 0.05, 'max_depth': 5},
   'rmse': np.float64(28.890075496370503),
   'mae': np.float64(14.765108021964611),
   'r2': np.float64(-0.09211156935411235)},
  {'params': {'learning_rate': 0.2, 'max_depth': 3},
   'rmse': np.float64(28.787906340263667),
   'mae': np.float64(15.894646208673336),
   'r2': np.float64(-0.0738080227004321)},
  {'params': {'learning_rate': 0.2, 'max_depth': 5},
   'rmse': np.float64(31.071348921738103),
   'mae': np.float64(17.276857030596627),
   'r2': np.float64(-0.49197932515681997)}]}

## Wrap-up

Write down: the best hyperparameters found, the RMSE gap between tuned and untuned `XGBRegressor`, and in one sentence whether a more complex model was worth it here compared to Module 4's `LinearRegression`.

_Your observations here._